In [1]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

This tutorial sets up a simulation of a short helical peptide with Chemical Shift restraints applied on top of the CHARMM27 forcefield. 

Staring from an extended conformation, the CS restraints should guide the system to a helical state.

## Define simulation parameters

In [2]:
temperature = 298*unit.kelvin
timestep = 2*unit.femtosecond
pressure = 1*unit.bar
collision_frequency = 1/unit.picosecond

max_K = 25
K_gradient = 0.001

report_interval = 1000

## Set up CHARMM27 system

In [3]:
gro = app.GromacsGroFile(f'Data/ShortHelix/system.gro')
top = app.GromacsTopFile(f'Data/ShortHelix/system.top', periodicBoxVectors=gro.getPeriodicBoxVectors(),
        includeDir=f'Data/GromacsForceFields/top')
system = top.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1*unit.nanometer,
        constraints=app.HBonds)

system.addForce(mm.AndersenThermostat(temperature, collision_frequency))
system.addForce(mm.MonteCarloBarostat(pressure, temperature))

8

## Add Chemical Shift restraints

In [4]:
napshift_force = get_napshift_force(top.topology, 'Data/ShortHelix/CS.txt', model_type='all_atom')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

/home/mina/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/BuildOpe

9

## Create the OpenMM simulation

In [ ]:
integrator = mm.VerletIntegrator(timestep)
platform = mm.Platform.getPlatformByName("CUDA")
simulation = app.Simulation(top.topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})

simulation.context.setPositions(gro.positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

## Add reporters

In [6]:
xtc_reporter = app.XTCReporter('Data/ShortHelix/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in top.topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

## Simulate!

In [8]:
warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {warmup_steps} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

Warming up CS restraints for 25000 steps
25000,50.00000000001514,-445539.73553387076,78448.0443414205,-367091.69119245023,297.52088423930127,319.2938751885635,39.9
26000,52.000000000017586,-446271.2506636246,78731.35769515633,-367539.89296846825,298.59537424383774,318.33808825798076,41.3
27000,54.00000000002003,-445432.7489211031,77718.01751109774,-367714.73141000536,294.75219535866137,319.4414699437957,42.5
28000,56.000000000022474,-445459.7308300771,78151.04276787535,-367308.6880622017,296.3944804962451,319.6759418158299,43.8
29000,58.00000000002492,-447127.6017801503,79235.46414705247,-367892.1376330978,300.50724086305036,319.1088115394791,45
30000,60.00000000002736,-446527.4238962624,78808.36649413453,-367719.0574021279,298.8874366675546,318.62987824302047,46.3
31000,62.00000000002981,-446488.30162889976,79371.0436568535,-367117.25797204627,301.0214376920392,319.39842509270557,47.5
32000,64.00000000003224,-445952.81409491994,79283.13897326081,-366669.6751216591,300.68805169866187,3

The output trajectory will be written to Data/ShortHelix